# APP

In [1]:
import os
from dotenv import load_dotenv
from groq import Groq
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [5]:
if os.path.exists('../.env'):
    load_dotenv()
    print("[INFO] Environment variables were loaded.")
else:
    print("[WARNING] File .env was not found. Some settings may be missing.")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("[ERROR] GROQ_API_KEY was not found in .env!")

[INFO] Environment variables were loaded.


## Utils

### Embeddings

In [6]:
import os
from langchain_huggingface import HuggingFaceEmbeddings

os.environ["TOKENIZERS_PARALLELISM"] = "false"

class Embeddings:
    def __init__(self):
        model_name = "BAAI/bge-base-en"
        encode_kwargs = {'normalize_embeddings': True} 
        self.model = HuggingFaceEmbeddings(
            model_name = model_name, 
            model_kwargs={'device': 'cpu'},
            encode_kwargs = encode_kwargs
            )

    def get_embedding_model(self):
        return self.model

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Retriever

In [8]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS

class Retriever:
    def __init__(self, embedding):
        self.embedding = embedding.get_embedding_model()
        self.documents = []
        self.vector_store = None

    def load_documents(self, documents_path):
        for filename in os.listdir(documents_path):
            if filename.endswith(".pdf"):
                loader = PyMuPDFLoader(os.path.join(documents_path, filename))
                loaded_docs = loader.load()
                self.documents.extend(loaded_docs)

    def create_vectordb(self):
        splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
        docs_split = splitter.split_documents(self.documents)
        self.vector_store = FAISS.from_documents(docs_split, self.embedding)

    def retrieve(self, query, k = 5):
        if not self.vector_store:
            raise ValueError("Vector store is not initialized.")
        
        docs = self.vector_store.similarity_search(query, k = k)
        
        return docs

---

## Main Classes

### DocumentAgent

In [ ]:
from groq import Groq

class DocumentAgent:
    def __init__(self, api_key):
        self.client = Groq(api_key = api_key)

    def summarize_documents(self, documents, query):
        context = "\n\n".join([doc.page_content for doc in documents])

        messages = [
            {"role": "system", "content": "You summarize documents with precision."},
            {"role": "user", "content": f"Documents: {context}\n\nAnswer the question briefly: {query}"}
        ]

        completion = self.client.chat.completions.create(model = "openai/gpt-oss-120b",
                                                         messages = messages,
                                                         temperature = 0.7,
                                                         max_tokens = 2048)
        
        

        return completion.choices[0].message.content.strip()

### ReasoningAgent

In [ ]:
from groq import Groq

class ReasoningAgent:
    def __init__(self, api_key):
        self.client = Groq(api_key = api_key)

    def generate_reasoning(self, summary, query):
        messages = [
            {"role": "system", "content": "You are a specialist in logical reasoning about texts."},
            {"role": "user", "content": f"Based on the summary: {summary}\n\nDo a critical analyse to answer the question:: {query}"}
        ]

        completion = self.client.chat.completions.create(model = "openai/gpt-oss-120b",
                                                         messages = messages,
                                                         temperature = 0.7,
                                                         max_tokens = 2048)
        return completion.choices[0].message.content.strip()

### MetaAgent

In [12]:
from groq import Groq

class MetaAgent:
    def __init__(self, api_key):
        self.client = Groq(api_key = api_key)

    def generate_final_answer(self, summary, reasoning, query):
        messages = [
            {"role": "system", "content": "You generate clear and detailed answers consolidating the information."},
            {"role": "user", "content": f"Original question: {query}\n\nSummary: {summary}\n\nLogical Reasoning: {reasoning}\n\nProvide the consolidated and detailed answer:"}
        ]

        completion = self.client.chat.completions.create(model = "openai/gpt-oss-120b",
                                                         messages = messages,
                                                         temperature = 0.7,
                                                         max_tokens = 2048,
                                                         stream = True)

        final_answer = ""

        for chunk in completion:
            chunk_content = chunk.choices[0].delta.content or ""
            final_answer += chunk_content

        return final_answer
